In [33]:
import pandas as pd

class CallCenter:
    def __init__(self):
        self.risks = {}
        self.sessions = None

    def load(self, path):
        self.sessions = pd.read_parquet(path)

    def preparing(self, df, is_train=True):
        merged = pd.merge(df, self.sessions, on='sessionkey_id', how='left')
        
        merged['is_payment'] = merged['page_type'] == 'payment'
        merged['is_error'] = merged['page_type'] == 'error'

        agg_cols = {
            'total_pages': ('page_type', 'count'),
            'has_payment_page': ('is_payment', 'max'),
            'has_error_page': ('is_error', 'max'),
            'max_price': ('price', 'max')
        }
        
        if is_train:
            agg_cols['is_callcenter'] = ('is_callcenter', 'first')

        agg = merged.groupby(['order_id']).agg(**agg_cols).reset_index()
        
        return agg

    def fit(self, df):
        self.risks['base'] = df['is_callcenter'].mean()
        
        if df['has_error_page'].sum() > 0:
            self.risks['error'] = df[df['has_error_page']]['is_callcenter'].mean()
        else:
            self.risks['error'] = self.risks['base']
            
        mask_few = df['total_pages'] < 2
        if mask_few.sum() > 0:
            self.risks['few'] = df[mask_few]['is_callcenter'].mean()
        else:
            self.risks['few'] = self.risks['base']

        mask_conf = (df['total_pages'] < 4) & (~df['has_payment_page'])
        if mask_conf.sum() > 0:
            self.risks['confused'] = df[mask_conf]['is_callcenter'].mean()
        else:
            self.risks['confused'] = self.risks['base']

        mask_exp = (df['max_price'] > 1500) & (~df['has_error_page'])
        if mask_exp.sum() > 0:
            self.risks['expensive'] = df[mask_exp]['is_callcenter'].mean()
        else:
            self.risks['expensive'] = self.risks['base']

        print("Вероятности из train:")
        for k, v in self.risks.items():
            print(f"{k}: {v:.2%}")

    def predict_prob(self, df):
        proba = pd.Series(self.risks['base'], index=df.index)
        
        mask_err = df['has_error_page']
        mask_few = df['total_pages'] < 2
        mask_conf = (df['total_pages'] < 4) & (~df['has_payment_page'])
        mask_exp = (df['max_price'] > 1500) & (~mask_err)

        proba[mask_err] = self.risks['error']
        proba[mask_few] = self.risks['few']
        proba[mask_conf] = self.risks['confused']
        proba[mask_exp] = self.risks['expensive']
        
        return proba

    def save(self, df, probs, filename):
        res = pd.DataFrame({'order_id': df['order_id'], 'is_callcenter': probs.values})
        res.to_csv(filename, index=False)
        return res

predictor = CallCenter()
predictor.load('sessions.parquet')

train = pd.read_parquet('train.parquet')
train_feat = predictor.preparing(train, is_train=True)
print(f"Train shape: {train_feat.shape}")
predictor.fit(train_feat)

test = pd.read_parquet('test.parquet')
test_feat = predictor.preparing(test, is_train=False)
print(f"Test shape: {test_feat.shape}")

probs = predictor.predict_prob(test_feat)
submission = predictor.save(test_feat, probs, 'my_submission.csv')

print(submission.head())

Train shape: (104595, 6)
Вероятности из train:
base: 35.47%
error: 35.47%
few: 66.43%
confused: 53.70%
expensive: 41.76%
Test shape: (17196, 5)
   order_id  is_callcenter
0   1340768       0.417576
1   1340769       0.354692
2   1340770       0.354692
3   1340772       0.354692
4   1340773       0.354692
